In [ ]:
import json
import re
from dataclasses import dataclass
import pandas as pd
import nltk
from transformers import AutoTokenizer
from langdetect import detect, LangDetectException

In [ ]:
# One-time download, cached after first run. Do this once outside the hot
# path -- don't call nltk.download() inside a loop over 1300 docs.
nltk.download("punkt_tab", quiet=True)
from nltk.tokenize import sent_tokenize

In [ ]:
def is_english(text: str, min_chars: int = 40) -> bool:
    """
    langdetect needs a reasonable amount of text to be reliable -- very
    short strings (a few words) can misfire. Below min_chars, default to
    True (don't drop short fragments on a shaky language call) and let
    length-based logic elsewhere handle them.

    NOTE: langdetect is a new dependency. If you'd rather not add it, a
    cheaper (rougher) proxy: check what fraction of characters fall in
    CJK Unicode ranges and threshold on that. Included below as a
    fallback in case you skip the langdetect install.
    """
    if len(text) < min_chars:
        return True
    try:
        return detect(text) == "en"
    except LangDetectException:
        return True  # detection failed -- don't drop on uncertainty


CJK_RE = re.compile(
    r"[\u4e00-\u9fff\u3040-\u30ff\uac00-\ud7af]"  # CJK unified, kana, hangul
)

def is_likely_non_english_cjk(text: str, threshold: float = 0.15) -> bool:
    """Dependency-free fallback if you skip langdetect."""
    if not text:
        return False
    cjk_chars = len(CJK_RE.findall(text))
    return (cjk_chars / max(len(text), 1)) > threshold

In [ ]:
"""
Updates to the chunking pipeline:
1. looks_like_glossary_or_metadata() -- catches abbreviation glossaries and
   editor/reviewer credit blocks that leaked into body sections (confirmed
   real examples: a "conclusion" that was actually an AIDA/Akt/ALS glossary,
   a "discussion" that was actually editor credits + an abbreviation key).
2. NLTK sent_tokenize replacing the regex splitter -- confirmed the regex
   was failing on dense technical prose (chemistry Methods example with
   semicolon-delimited clauses and decimal-point-heavy nomenclature).
"""


MEDCPT_MODEL_NAME = "ncbi/MedCPT-Article-Encoder"
TARGET_TOKENS = 400
MAX_TOKENS = 480
OVERLAP_TOKENS = 70

ATOMIC_CANONICAL_TYPES = {
    "abstract", "figure_footnote", "data_availability",
    "title", "keywords",
}


# ---------------------------------------------------------------------------
# Fix A: glossary / metadata leakage detector
# ---------------------------------------------------------------------------

GLOSSARY_PATTERN_RE = re.compile(r"\b[A-Z][A-Za-z0-9\-]{1,8}:\s")
SENTENCE_ENDER_RE = re.compile(r"[.!?]\s+[A-Z]")


def looks_like_glossary_or_metadata(text: str) -> bool:
    """
    Heuristic, not a certainty -- catches "Term: definition" glossary
    blocks and editor/reviewer credit lines. Confirmed against real
    examples (AIDA/Akt/ALS abbreviation glossary tagged as 'conclusion';
    'Edited by:... Reviewed by:...' tagged as 'discussion'). Tune the
    threshold below against your own false-positive/negative rate --
    don't trust `>= 3` as gospel, it's a starting point.
    """
    if not text:
        return False
    glossary_hits = len(GLOSSARY_PATTERN_RE.findall(text))
    sentence_enders = len(SENTENCE_ENDER_RE.findall(text))
    return glossary_hits >= 3 and glossary_hits > sentence_enders


# ---------------------------------------------------------------------------
# Fix B: real sentence tokenizer instead of hand-rolled regex
# ---------------------------------------------------------------------------

def split_sentences(text: str) -> list[str]:
    """
    NLTK's punkt handles abbreviations, decimals, and common scientific
    punctuation patterns far better than a regex lookahead did -- but it's
    still a general-purpose English tokenizer, not biomedical-specific.
    If dense chemistry/genomics text (like the LC-MS procedure example)
    still produces bad splits after this, scispacy's sentence splitter
    (trained on biomedical text) is the next upgrade to reach for.
    """
    sentences = sent_tokenize(text.strip())
    return [s.strip() for s in sentences if s.strip()]


@dataclass
class TokenizedSentence:
    text: str
    n_tokens: int


class MedCPTChunker:
    def __init__(self, model_name: str = MEDCPT_MODEL_NAME):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)

    def count_tokens(self, text: str) -> int:
        return len(self.tokenizer.encode(text, add_special_tokens=False))

    def chunk_section(self, section: dict) -> list[dict]:
        text = section["text"]
        canonical = section["canonical_section_type"]

        if is_likely_non_english_cjk(text):  # or: not is_english(text)
            return [{
                "text": text,
                "n_tokens": self.count_tokens(text),
                "reclassified_as": "non_english_dropped",
                "keep": False,
            }]

        if looks_like_glossary_or_metadata(text):
            return [{
                "text": text,
                "n_tokens": self.count_tokens(text),
                "reclassified_as": "glossary_or_metadata",
                "keep": False,
            }]

        total_tokens = self.count_tokens(text)

        if total_tokens <= MAX_TOKENS:
            return [{"text": text, "n_tokens": total_tokens, "keep": True}]

        if canonical in ATOMIC_CANONICAL_TYPES:
            return self._hard_split_no_overlap(text)

        return self._sliding_window_chunk(text)

    def _hard_split_no_overlap(self, text: str) -> list[dict]:
        sentences = split_sentences(text)
        if not sentences:
            return [{"text": text, "n_tokens": self.count_tokens(text)}]

        lengths = [
            len(ids) for ids in
            self.tokenizer(sentences, add_special_tokens=False)["input_ids"]
        ]
        chunks, current, current_tokens = [], [], 0

        for s, n in zip(sentences, lengths):
            if current_tokens + n > MAX_TOKENS and current:
                chunks.append({"text": " ".join(current), "n_tokens": current_tokens})
                current, current_tokens = [], 0
            current.append(s)
            current_tokens += n

        if current:
            chunks.append({"text": " ".join(current), "n_tokens": current_tokens})

        return chunks

    def _sliding_window_chunk(self, text: str) -> list[dict]:
        """
        Sentence-aware sliding window with bounded overlap. Single-pass
        `for` loop (not manually-indexed `while`) so it structurally
        cannot infinite-loop -- fixed per your earlier catch. Includes
        your step-3 overlap trimming so overlap+next-sentence never
        exceeds MAX_TOKENS.
        """
        sentences = split_sentences(text)
        if not sentences:
            return [{"text": text, "n_tokens": self.count_tokens(text)}]

        lengths = [
            len(ids) for ids in
            self.tokenizer(sentences, add_special_tokens=False)["input_ids"]
        ]
        tokenized = [TokenizedSentence(s, n) for s, n in zip(sentences, lengths)]

        chunks: list[dict] = []
        current: list[TokenizedSentence] = []
        current_tokens = 0

        for sent in tokenized:
            if sent.n_tokens > MAX_TOKENS:
                if current:
                    chunks.append(self._finalize(current))
                    current, current_tokens = [], 0
                chunks.append(self._finalize([sent]))
                continue

            if current_tokens + sent.n_tokens > MAX_TOKENS:
                if current:
                    chunks.append(self._finalize(current))

                overlap_sents: list[TokenizedSentence] = []
                overlap_tok_count = 0
                for prev_sent in reversed(current):
                    if overlap_tok_count + prev_sent.n_tokens > OVERLAP_TOKENS:
                        break
                    overlap_sents.insert(0, prev_sent)
                    overlap_tok_count += prev_sent.n_tokens

                current = overlap_sents
                current_tokens = overlap_tok_count

                while current_tokens + sent.n_tokens > MAX_TOKENS and current:
                    dropped = current.pop(0)
                    current_tokens -= dropped.n_tokens

            current.append(sent)
            current_tokens += sent.n_tokens

        if current:
            chunks.append(self._finalize(current))

        return chunks

    def _finalize(self, sentences: list["TokenizedSentence"]) -> dict:
        text = " ".join(s.text for s in sentences)
        n_tokens = sum(s.n_tokens for s in sentences)

        if n_tokens > MAX_TOKENS:
            # Last-resort safety net for confirmed-junk residual (dense
            # chemistry nomenclature, abbreviation tables that slipped past
            # the glossary heuristic, oversized footnotes). We've already
            # decided this content is low-value -- truncating it is an
            # acceptable loss here, NOT a strategy for real prose. If this
            # branch starts firing on legitimate methods/results content,
            # that's a signal to go fix the actual splitter/classifier,
            # not a green light to lean on truncation more broadly.
            token_ids = self.tokenizer.encode(text, add_special_tokens=False)
            truncated_ids = token_ids[:MAX_TOKENS]
            text = self.tokenizer.decode(truncated_ids, skip_special_tokens=True)
            n_tokens = MAX_TOKENS

        return {"text": text, "n_tokens": n_tokens, "truncated": n_tokens == MAX_TOKENS}

In [ ]:
def build_parents_and_children(doc: dict, chunker: MedCPTChunker) -> tuple[list[dict], list[dict]]:
    parents = []
    children = []

    doc_meta = {
        "doc_id": doc["doc_id"],
        "pmc_id": doc["pmc_id"],
        "doi": doc["doi"],
        "year": doc["year"],
        "kwd": doc["kwd"],
        "title": doc["title"],
    }

    for section in doc["sections"]:
        parent_id = section["section_id"]
        parents.append({
            "parent_id": parent_id,
            **doc_meta,
            "heading_path": section["heading_path"],
            "canonical_section_type": section["canonical_section_type"],
            "text": section["text"],
            "n_words": section["n_words"],
        })

        child_dicts = chunker.chunk_section(section)
        for idx, c in enumerate(child_dicts):
            # if the glossary/metadata detector fired, the CHILD's
            # canonical_section_type is overridden -- the PARENT record
            # above deliberately keeps its original classification, so
            # you can still audit what got reclassified vs. what didn't
            effective_canonical = c.get("reclassified_as", section["canonical_section_type"])

            children.append({
                "chunk_id": f"{parent_id}_c{idx}",
                "parent_id": parent_id,
                **doc_meta,
                "heading_path": section["heading_path"],
                "canonical_section_type": effective_canonical,
                "original_canonical_section_type": section["canonical_section_type"],
                "text": c["text"],
                "n_tokens": c["n_tokens"],
                "chunk_index": idx,
                "n_chunks_in_parent": len(child_dicts),
            })

    return parents, children

In [ ]:
def load_cleaned_jsonl(path: str):
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                yield json.loads(line)


def save_jsonl(records: list[dict], path: str) -> None:
    with open(path, "w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

In [ ]:
if __name__ == "__main__":
    CLEAN_PATH = "/home/razor/Documents/alzheimers-rag/data/processed/pmc_alzheimers_cleaned.jsonl"
    PARENTS_PATH = "/home/razor/Documents/alzheimers-rag/data/processed/pmc_alzheimers_parents.jsonl"
    CHILDREN_PATH = "/home/razor/Documents/alzheimers-rag/data/processed/pmc_alzheimers_children.jsonl"

    chunker = MedCPTChunker()

    all_parents, all_children = [], []
    for doc in load_cleaned_jsonl(CLEAN_PATH):
        p, c = build_parents_and_children(doc, chunker)
        all_parents.extend(p)
        all_children.extend(c)

    save_jsonl(all_parents, PARENTS_PATH)
    final_children = [c for c in all_children if not c.get("canonical_section_type") in {"glossary_or_metadata", "non_english_dropped"}]
    save_jsonl(final_children, CHILDREN_PATH)

    print(f"{len(all_parents)} parent sections -> {PARENTS_PATH}")
    print(f"{len(all_children)} child chunks -> {CHILDREN_PATH}")
    print(f"avg children per parent: {len(final_children)/max(len(all_parents),1):.2f}")

In [ ]:
token_count_dist = pd.DataFrame({"n_tokens": [c["n_tokens"] for c in final_children]})

In [ ]:
token_count_dist.hist(bins=40, figsize=(10, 6), grid=False, color='skyblue')

In [ ]:
token_count_dist[token_count_dist["n_tokens"] > 512].describe()